# 🧠 ForgetLab: Cross-Lingual Machine Unlearning Studio on Kaggle (T4 GPU)

This notebook sets up the ForgetLab environment, prepares the injected model, runs verification, and launches the Gradio dashboard with a public URL (`share=True`).

### Prerequisites:
1. **Kaggle Accelerator**: Set to **GPU T4** or **GPU T4 x 2** (Settings panel on right).
2. **Internet Access**: Set to **Internet On** (Settings panel on right).
3. **HuggingFace Token**: Add your HF token to **Kaggle Secrets** (Add-ons -> Secrets) with label `HF_TOKEN`.

In [ ]:
# ── Cell 1: Environment Setup, Dependencies & Auth ──────────────────────────
import os, sys, subprocess
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# 1. Install dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.46.0", "peft==0.13.2", "trl==0.11.4",
    "accelerate==1.0.1", "bitsandbytes>=0.45.3",
    "datasets==3.0.1", "sacrebleu", "scikit-learn",
    "gradio>=4.0", "sacremoses", "pandas", "matplotlib",
    "git+https://github.com/VarunGumma/IndicTransToolkit.git"
], check=True)

# 2. Clone or update repository
REPO_DIR = "/kaggle/working/Computing-Lab-Project"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/InsaneNitish/Computing-Lab-Project.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 3. Authenticate with HuggingFace
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("✅ HuggingFace authentication successful.")
except Exception as e:
    print("⚠️ HuggingFace login warning:", e)
    print("Make sure HF_TOKEN secret is set in Kaggle if loading gated Llama-3.2-1B.")

print("\nReady for model setup!")

In [ ]:
# ── Cell 2: Knowledge Injection (Build checkpoints/injected) ───────────────
import os, subprocess, sys

injected_config = "checkpoints/injected/adapter_config.json"
if not os.path.exists(injected_config):
    print("Running knowledge injection (training Llama-3.2-1B on TOFU facts across English, Hindi, Kannada)…")
    subprocess.run([
        sys.executable, "-m", "src.training.inject_knowledge",
        "--langs", "english", "hindi", "kannada",
        "--epochs", "15",
        "--lr", "2e-4",
        "--bs", "2",
        "--accum", "4",
        "--max_len", "768",
        "--out", "checkpoints/injected",
    ], check=True)
    print("✅ Knowledge injection completed successfully!")
else:
    print("✅ checkpoints/injected/ already exists — skipping re-training.")

In [ ]:
# ── Cell 3: Baseline Verification ──────────────────────────────────────────
import os, subprocess, sys, json

print("Evaluating baseline knowledge on forget and retain sets across EN, HI, KN…")
subprocess.run([
    sys.executable, "-m", "src.eval.transfer_matrix",
    "--query_langs", "english", "hindi", "kannada",
    "--n", "50",
    "--injected", "checkpoints/injected",
    "--baseline",
    "--out", "results/transfer_matrix.json",
], check=True)

with open("results/transfer_matrix.json", encoding="utf-8") as f:
    tm = json.load(f)
print("\n=== BASELINE RESULTS ===")
print(json.dumps(tm.get("baseline", {}), indent=2))
print("✅ Baseline verification complete!")

In [ ]:
# ── Cell 4: Launch ForgetLab Public Dashboard ──────────────────────────────
# Running this cell starts the Gradio web server with a public live link!
from forgetlab.app import launch
launch(share=True, port=7860)